In [1]:
import sys
import platform
import pandas as pd
import numpy as np
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
Pandas: 3.0.5
NumPy: 2.4.6
Scikit-learn: 1.9.0


In [2]:
print("PoisonShield notebook is working.")

PoisonShield notebook is working.


In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DIR)

print("\nFiles found:")
for file in RAW_DIR.iterdir():
    print("-", file.name)

Project root:
d:\User\Desktop\PoisonShield

Raw data directory:
d:\User\Desktop\PoisonShield\data\raw

Files found:
- cccs_andmal2020_poisoned.csv
- cccs_andmal2020_removed_audit.csv
- cic_malmem2022_poisoned.csv
- cic_malmem2022_removed_audit.csv


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


Phase - 2 : Dataset Preparation & Audit

Locate the project root safely

In [5]:
from pathlib import Path

cwd = Path.cwd()

print("Current working directory:")
print(cwd)

print("\nDirectory contents:")
for item in cwd.iterdir():
    print("-", item.name)

Current working directory:
d:\User\Desktop\PoisonShield\notebooks

Directory contents:
- 01_dataset_audit.ipynb


Automatically find the project root

In [6]:
from pathlib import Path

def find_project_root(start_path):
    start_path = Path(start_path).resolve()

    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (
            (path / "data" / "raw").exists()
            and (path / "notebooks").exists()
            and (path / "src").exists()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate the PoisonShield project root."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DIR)

Project root:
D:\User\Desktop\PoisonShield

Raw data directory:
D:\User\Desktop\PoisonShield\data\raw


Verify the four expected files

In [8]:
expected_files = [
    "cic_malmem2022_poisoned.csv",
    "cic_malmem2022_removed_audit.csv",
    "cccs_andmal2020_poisoned.csv",
    "cccs_andmal2020_removed_audit.csv",
]

print("Expected raw files:\n")

for filename in expected_files:
    path = RAW_DIR / filename

    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"[FOUND] {filename} — {size_mb:.2f} MB")
    else:
        print(f"[MISSING] {filename}")

Expected raw files:

[FOUND] cic_malmem2022_poisoned.csv — 16.97 MB
[FOUND] cic_malmem2022_removed_audit.csv — 0.89 MB
[FOUND] cccs_andmal2020_poisoned.csv — 22.02 MB
[FOUND] cccs_andmal2020_removed_audit.csv — 1.49 MB


Check that raw files have not been accidentally modified

In [9]:
print("RAW DATA FILE INVENTORY")
print("=" * 70)

for filename in expected_files:
    path = RAW_DIR / filename

    if path.exists():
        stat = path.stat()

        print(f"\nFile: {filename}")
        print(f"Size: {stat.st_size:,} bytes")
        print(f"Path: {path}")

RAW DATA FILE INVENTORY

File: cic_malmem2022_poisoned.csv
Size: 17,793,118 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cic_malmem2022_poisoned.csv

File: cic_malmem2022_removed_audit.csv
Size: 929,617 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cic_malmem2022_removed_audit.csv

File: cccs_andmal2020_poisoned.csv
Size: 23,087,296 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cccs_andmal2020_poisoned.csv

File: cccs_andmal2020_removed_audit.csv
Size: 1,563,080 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cccs_andmal2020_removed_audit.csv


Tables 

In [10]:
inventory_rows = []

for filename in expected_files:
    path = RAW_DIR / filename

    inventory_rows.append({
        "filename": filename,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan
    })

file_inventory = pd.DataFrame(inventory_rows)

file_inventory

,filename,exists,size_bytes
0,cic_malmem2022_poisoned.csv,True,17793118
1,cic_malmem2022_removed_audit.csv,True,929617
2,cccs_andmal2020_poisoned.csv,True,23087296
3,cccs_andmal2020_removed_audit.csv,True,1563080


In [11]:
inventory_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_raw_file_inventory.csv"
)

file_inventory.to_csv(inventory_path, index=False)

print(f"Saved inventory to:\n{inventory_path}")

Saved inventory to:
D:\User\Desktop\PoisonShield\results\tables\table_raw_file_inventory.csv


Define the two paths

In [12]:
malmem_path = RAW_DIR / "cic_malmem2022_poisoned.csv"
android_path = RAW_DIR / "cccs_andmal2020_poisoned.csv"

print("MalMem path:")
print(malmem_path)

print("\nAndroid path:")
print(android_path)

MalMem path:
D:\User\Desktop\PoisonShield\data\raw\cic_malmem2022_poisoned.csv

Android path:
D:\User\Desktop\PoisonShield\data\raw\cccs_andmal2020_poisoned.csv


Confirm the files exist

In [13]:
assert malmem_path.exists(), f"Missing file: {malmem_path}"
assert android_path.exists(), f"Missing file: {android_path}"

print("Both primary datasets were found successfully.")

Both primary datasets were found successfully.


Load the datasets

In [14]:
malmem = pd.read_csv(malmem_path)
android = pd.read_csv(android_path)

print("Datasets loaded successfully.")

Datasets loaded successfully.


Verify dimensions

In [15]:
print("Dataset shapes")
print("=" * 50)

print(f"CIC-MalMem-2022: {malmem.shape}")
print(f"Android dataset: {android.shape}")

Dataset shapes
CIC-MalMem-2022: (58596, 59)
Android dataset: (11598, 474)


In [16]:
expected_shapes = {
    "MalMem": (58596, 59),
    "Android": (11598, 474),
}

actual_shapes = {
    "MalMem": malmem.shape,
    "Android": android.shape,
}

for name in expected_shapes:
    expected = expected_shapes[name]
    actual = actual_shapes[name]

    status = "PASS" if expected == actual else "FAIL"

    print(
        f"{name}: {status} | "
        f"Expected={expected}, Actual={actual}"
    )

MalMem: PASS | Expected=(58596, 59), Actual=(58596, 59)
Android: PASS | Expected=(11598, 474), Actual=(11598, 474)


Inspect the first five rows

In [17]:
print("CIC-MalMem-2022")
display(malmem.head())

print("\nSupplied Android dataset")
display(android.head())

CIC-MalMem-2022


,Class,pslist.nproc,pslist.nppid,pslist.avg_threads,pslist.nprocs64bit,pslist.avg_handlers,dlllist.ndlls,dlllist.avg_dlls_per_proc,handles.nhandles,handles.avg_handles_per_proc,...,svcscan.process_services,svcscan.shared_process_services,svcscan.interactive_process_services,svcscan.nactive,callbacks.ncallbacks,callbacks.nanonymous,callbacks.ngeneric,attack_type,original_label,source_index
0,Benign,45,17,10.555556,0,202.844444,1694.0,38.500000,9129.0,212.302326,...,24,116,0,121.0,87.0,0,8,clean,Benign,-1
1,Benign,47,19,11.531915,0,242.234043,2074.0,44.127660,11385.0,242.234043,...,24,118,0,122.0,87.0,0,8,clean,Benign,-1
2,Benign,40,14,14.725000,0,288.225000,1932.0,48.300000,11529.0,288.225000,...,27,118,0,120.0,88.0,0,8,clean,Benign,-1
3,Benign,32,13,13.500000,0,264.281250,1445.0,45.156250,8457.0,264.281250,...,27,118,0,120.0,88.0,0,8,clean,Benign,-1
4,Benign,42,16,11.452381,0,281.333333,2067.0,49.214286,11816.0,281.333333,...,24,118,0,124.0,87.0,0,8,clean,Benign,-1



Supplied Android dataset


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev,Class,attack_type,original_label,source_index
0,1.0,0.0,0.0,3.0,0.0,14.0,2.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,37.0,10.000000,Benign,backdoor_injection,Adware,-1
1,3.0,0.0,0.0,6.0,0.0,42.0,91.0,0.0,32.0,0.0,...,0.0,0.0,0.0,2.0,2838.0,7.257753,Adware,feature_poisoning,Adware,-1
2,2.0,0.0,0.0,4.0,0.0,23.0,3.0,0.0,17.0,2.0,...,0.0,0.0,0.0,1.0,111.0,2.776177,Adware,feature_poisoning,Adware,-1
3,3.0,0.0,0.0,11.0,0.0,18.0,3.0,0.0,16.0,0.0,...,0.0,0.0,0.0,1.0,98.0,25.000000,Adware,clean,Adware,-1
4,0.0,0.0,0.0,0.0,0.0,9.0,2.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,60.0,3.000000,Adware,clean,Adware,-1


Inspect the last five rows

In [18]:
print("CIC-MalMem-2022 — last 5 rows")
display(malmem.tail())

print("\nAndroid dataset — last 5 rows")
display(android.tail())

CIC-MalMem-2022 — last 5 rows


,Class,pslist.nproc,pslist.nppid,pslist.avg_threads,pslist.nprocs64bit,pslist.avg_handlers,dlllist.ndlls,dlllist.avg_dlls_per_proc,handles.nhandles,handles.avg_handles_per_proc,...,svcscan.process_services,svcscan.shared_process_services,svcscan.interactive_process_services,svcscan.nactive,callbacks.ncallbacks,callbacks.nanonymous,callbacks.ngeneric,attack_type,original_label,source_index
58591,Spyware,42,18,9.714286,0,202.738095,1596.0,38.000000,8515.0,202.738095,...,24,116,0,119.0,86.0,0,8,sample_duplication,Spyware,44788
58592,Spyware,35,15,10.514286,0,184.485714,1273.0,36.371429,6458.0,189.941177,...,21,110,0,112.0,88.0,0,8,sample_duplication,Spyware,38485
58593,Spyware,40,16,9.850000,0,213.850000,1547.0,38.675000,8555.0,219.358974,...,24,116,0,119.0,86.0,0,8,sample_duplication,Spyware,45066
58594,Spyware,40,16,9.575000,0,204.250000,1506.0,37.650000,8171.0,209.512821,...,24,116,0,119.0,87.0,0,8,sample_duplication,Spyware,43756
58595,Spyware,40,16,9.825000,0,208.675000,1557.0,38.925000,8347.0,208.675000,...,24,116,0,122.0,86.0,0,8,sample_duplication,Spyware,36374



Android dataset — last 5 rows


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev,Class,attack_type,original_label,source_index
11593,0.0,0.0,0.0,3.0,0.0,10.0,2.0,0.0,22.0,0.0,...,0.0,0.0,0.0,1.0,1162.0,10.0,Riskware,sample_duplication,Riskware,8201
11594,0.0,0.0,0.0,5.0,0.0,36.0,24.0,0.0,47.0,3.0,...,0.0,0.0,0.0,2.0,2649.0,27.0,Riskware,sample_duplication,Riskware,7696
11595,216.0,0.0,0.0,31.0,19.0,95.0,190.0,32.0,359.0,26.0,...,0.0,162.0,0.0,1.0,1731.0,992.0,Riskware,sample_duplication,Riskware,8939
11596,1.0,0.0,0.0,11.0,1.0,13.0,26.0,1.0,30.0,0.0,...,0.0,1.0,0.0,1.0,287.0,138.0,Riskware,sample_duplication,Riskware,7500
11597,4.0,0.0,0.0,1.0,0.0,14.0,3.0,0.0,22.0,1.0,...,0.0,0.0,0.0,0.0,177.0,80.0,Riskware,sample_duplication,Riskware,7822


Inspect column names

In [19]:
print("CIC-MalMem-2022 columns")
print("=" * 70)

for i, col in enumerate(malmem.columns):
    print(f"{i:>3}: {col}")

print("\n\nAndroid dataset columns")
print("=" * 70)

for i, col in enumerate(android.columns):
    print(f"{i:>3}: {col}")

CIC-MalMem-2022 columns
  0: Class
  1: pslist.nproc
  2: pslist.nppid
  3: pslist.avg_threads
  4: pslist.nprocs64bit
  5: pslist.avg_handlers
  6: dlllist.ndlls
  7: dlllist.avg_dlls_per_proc
  8: handles.nhandles
  9: handles.avg_handles_per_proc
 10: handles.nport
 11: handles.nfile
 12: handles.nevent
 13: handles.ndesktop
 14: handles.nkey
 15: handles.nthread
 16: handles.ndirectory
 17: handles.nsemaphore
 18: handles.ntimer
 19: handles.nsection
 20: handles.nmutant
 21: ldrmodules.not_in_load
 22: ldrmodules.not_in_init
 23: ldrmodules.not_in_mem
 24: ldrmodules.not_in_load_avg
 25: ldrmodules.not_in_init_avg
 26: ldrmodules.not_in_mem_avg
 27: malfind.ninjections
 28: malfind.commitCharge
 29: malfind.protection
 30: malfind.uniqueInjections
 31: psxview.not_in_pslist
 32: psxview.not_in_eprocess_pool
 33: psxview.not_in_ethread_pool
 34: psxview.not_in_pspcid_list
 35: psxview.not_in_csrss_handles
 36: psxview.not_in_session
 37: psxview.not_in_deskthrd
 38: psxview.not_in_

Verify duplicate column names

In [20]:
def duplicate_column_names(df):
    return df.columns[df.columns.duplicated()].tolist()


malmem_duplicate_columns = duplicate_column_names(malmem)
android_duplicate_columns = duplicate_column_names(android)

print("MalMem duplicate column names:")
print(malmem_duplicate_columns)

print("\nAndroid duplicate column names:")
print(android_duplicate_columns)

MalMem duplicate column names:
[]

Android duplicate column names:
[]


Check column-name whitespace

In [21]:
def columns_with_whitespace(df):
    return [
        col for col in df.columns
        if col != col.strip()
    ]


print("MalMem columns with leading/trailing whitespace:")
print(columns_with_whitespace(malmem))

print("\nAndroid columns with leading/trailing whitespace:")
print(columns_with_whitespace(android))

MalMem columns with leading/trailing whitespace:
[]

Android columns with leading/trailing whitespace:
[]


Inspect data types

In [22]:
print("MalMem data types")
print("=" * 50)
print(malmem.dtypes.value_counts())

print("\nAndroid data types")
print("=" * 50)
print(android.dtypes.value_counts())

MalMem data types
int64      33
float64    23
str         3
Name: count, dtype: int64

Android data types
float64    470
str          3
int64        1
Name: count, dtype: int64


Explicitly identify numeric and non-numeric columns

In [23]:
def inspect_column_types(df, dataset_name):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    non_numeric_cols = df.select_dtypes(exclude=np.number).columns.tolist()

    print(f"\n{dataset_name}")
    print("=" * 70)

    print(f"Total columns: {len(df.columns)}")
    print(f"Numeric columns: {len(numeric_cols)}")
    print(f"Non-numeric columns: {len(non_numeric_cols)}")

    print("\nNon-numeric columns:")
    for col in non_numeric_cols:
        print(f"  - {col}")


inspect_column_types(malmem, "CIC-MalMem-2022")
inspect_column_types(android, "Supplied Android Dataset")


CIC-MalMem-2022
Total columns: 59
Numeric columns: 56
Non-numeric columns: 3

Non-numeric columns:
  - Class
  - attack_type
  - original_label

Supplied Android Dataset
Total columns: 474
Numeric columns: 471
Non-numeric columns: 3

Non-numeric columns:
  - Class
  - attack_type
  - original_label


Verify the four critical metadata fields

In [24]:
metadata_cols = [
    "Class",
    "attack_type",
    "original_label",
    "source_index",
]

for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    print(f"\n{name}")
    print("=" * 70)

    for col in metadata_cols:
        if col in df.columns:
            print(f"[FOUND]   {col:<20} dtype={df[col].dtype}")
        else:
            print(f"[MISSING] {col}")


MalMem
[FOUND]   Class                dtype=str
[FOUND]   attack_type          dtype=str
[FOUND]   original_label       dtype=str
[FOUND]   source_index         dtype=int64

Android
[FOUND]   Class                dtype=str
[FOUND]   attack_type          dtype=str
[FOUND]   original_label       dtype=str
[FOUND]   source_index         dtype=int64


Create a structural audit table

In [25]:
structural_audit = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "rows": malmem.shape[0],
        "columns": malmem.shape[1],
        "numeric_columns": malmem.select_dtypes(include=np.number).shape[1],
        "non_numeric_columns": malmem.select_dtypes(exclude=np.number).shape[1],
        "has_Class": "Class" in malmem.columns,
        "has_attack_type": "attack_type" in malmem.columns,
        "has_original_label": "original_label" in malmem.columns,
        "has_source_index": "source_index" in malmem.columns,
    },
    {
        "dataset": "Supplied Android Dataset",
        "rows": android.shape[0],
        "columns": android.shape[1],
        "numeric_columns": android.select_dtypes(include=np.number).shape[1],
        "non_numeric_columns": android.select_dtypes(exclude=np.number).shape[1],
        "has_Class": "Class" in android.columns,
        "has_attack_type": "attack_type" in android.columns,
        "has_original_label": "original_label" in android.columns,
        "has_source_index": "source_index" in android.columns,
    }
])

display(structural_audit)

,dataset,rows,columns,numeric_columns,non_numeric_columns,has_Class,has_attack_type,has_original_label,has_source_index
0,CIC-MalMem-2022,58596,59,56,3,True,True,True,True
1,Supplied Android Dataset,11598,474,471,3,True,True,True,True


In [26]:
structural_audit_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_structural_audit.csv"
)

structural_audit.to_csv(structural_audit_path, index=False)

print(f"Saved:\n{structural_audit_path}")

Saved:
D:\User\Desktop\PoisonShield\results\tables\table_structural_audit.csv


Construct poisoned and Validate Attack Composition

Inspect attack types

In [27]:
print("CIC-MalMem-2022 attack types")
print("=" * 70)
print(malmem["attack_type"].value_counts(dropna=False))

print("\n\nAndroid attack types")
print("=" * 70)
print(android["attack_type"].value_counts(dropna=False))

CIC-MalMem-2022 attack types
attack_type
clean                       42202
backdoor_injection           2342
gaussian_noise_injection     2342
label_flipping               2342
feature_poisoning            2342
missing_value_injection      2342
outlier_injection            2342
sample_duplication           2342
Name: count, dtype: int64


Android attack types
attack_type
clean                       6117
backdoor_injection           783
feature_poisoning            783
missing_value_injection      783
gaussian_noise_injection     783
label_flipping               783
outlier_injection            783
sample_duplication           783
Name: count, dtype: int64


Check missing attack labels

In [28]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    missing_attack_type = df["attack_type"].isna().sum()

    print(
        f"{name}: missing attack_type values = "
        f"{missing_attack_type}"
    )

MalMem: missing attack_type values = 0
Android: missing attack_type values = 0


Create the poisoned evaluation variable

In [29]:
malmem["poisoned"] = (
    malmem["attack_type"] != "clean"
).astype(int)

android["poisoned"] = (
    android["attack_type"] != "clean"
).astype(int)

C:\Users\alokr\AppData\Local\Temp\ipykernel_15052\3305891863.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  android["poisoned"] = (


Verify the new variable

In [30]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    print(f"\n{name}")
    print("=" * 70)

    print("Unique poisoned values:")
    print(sorted(df["poisoned"].unique()))

    print("\nCounts:")
    print(df["poisoned"].value_counts().sort_index())


MalMem
Unique poisoned values:
[np.int64(0), np.int64(1)]

Counts:
poisoned
0    42202
1    16394
Name: count, dtype: int64

Android
Unique poisoned values:
[np.int64(0), np.int64(1)]

Counts:
poisoned
0    6117
1    5481
Name: count, dtype: int64


Validate the definition mathematically

In [31]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    expected_poisoned = (
        df["attack_type"] != "clean"
    ).astype(int)

    mismatches = (
        df["poisoned"] != expected_poisoned
    ).sum()

    print(
        f"{name}: poisoned-definition mismatches = {mismatches}"
    )

MalMem: poisoned-definition mismatches = 0
Android: poisoned-definition mismatches = 0


Calculate poisoning proportion

In [32]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    poisoned_rate = df["poisoned"].mean() * 100

    print(
        f"{name}: {poisoned_rate:.3f}% of observations are poisoned"
    )

MalMem: 27.978% of observations are poisoned
Android: 47.258% of observations are poisoned


Build a complete attack-distribution table

In [33]:
attack_distribution = pd.concat(
    [
        (
            malmem["attack_type"]
            .value_counts()
            .rename_axis("attack_type")
            .reset_index(name="count")
            .assign(dataset="CIC-MalMem-2022")
        ),
        (
            android["attack_type"]
            .value_counts()
            .rename_axis("attack_type")
            .reset_index(name="count")
            .assign(dataset="Android")
        ),
    ],
    ignore_index=True
)

attack_distribution["percentage"] = (
    attack_distribution.groupby("dataset")["count"]
    .transform(lambda x: 100 * x / x.sum())
)

attack_distribution = attack_distribution[
    ["dataset", "attack_type", "count", "percentage"]
].sort_values(
    ["dataset", "attack_type"]
).reset_index(drop=True)

display(attack_distribution)

,dataset,attack_type,count,percentage
0,Android,backdoor_injection,783,6.751164
1,Android,clean,6117,52.741852
2,Android,feature_poisoning,783,6.751164
3,Android,gaussian_noise_injection,783,6.751164
4,Android,label_flipping,783,6.751164
5,Android,missing_value_injection,783,6.751164
6,Android,outlier_injection,783,6.751164
7,Android,sample_duplication,783,6.751164
8,CIC-MalMem-2022,backdoor_injection,2342,3.996860
9,CIC-MalMem-2022,clean,42202,72.021981


Add poisoned counts

In [34]:
poisoned_summary = []

for name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    poisoned_count = int(df["poisoned"].sum())
    clean_count = int((df["poisoned"] == 0).sum())
    total = len(df)

    poisoned_summary.append({
        "dataset": name,
        "total_observations": total,
        "clean_observations": clean_count,
        "poisoned_observations": poisoned_count,
        "clean_percentage": 100 * clean_count / total,
        "poisoned_percentage": 100 * poisoned_count / total
    })

poisoned_summary = pd.DataFrame(poisoned_summary)

display(poisoned_summary)

,dataset,total_observations,clean_observations,poisoned_observations,clean_percentage,poisoned_percentage
0,CIC-MalMem-2022,58596,42202,16394,72.021981,27.978019
1,Android,11598,6117,5481,52.741852,47.258148


Save the audit artifacts

In [35]:
attack_table_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_attack_distribution.csv"
)

poisoned_summary_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_poisoned_summary.csv"
)

attack_distribution.to_csv(
    attack_table_path,
    index=False
)

poisoned_summary.to_csv(
    poisoned_summary_path,
    index=False
)

print("Saved:")
print(attack_table_path)
print(poisoned_summary_path)

Saved:
D:\User\Desktop\PoisonShield\results\tables\table_attack_distribution.csv
D:\User\Desktop\PoisonShield\results\tables\table_poisoned_summary.csv


Create a validation report

In [36]:
validation_checks = []

for name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    attack_types = set(df["attack_type"].dropna().unique())

    expected_attack_types = {
        "clean",
        "backdoor_injection",
        "feature_poisoning",
        "gaussian_noise_injection",
        "label_flipping",
        "missing_value_injection",
        "sample_duplication",
        "outlier_injection"
    }

    validation_checks.append({
        "dataset": name,
        "attack_type_count": len(attack_types),
        "attack_types_correct": attack_types == expected_attack_types,
        "missing_attack_type": int(df["attack_type"].isna().sum()),
        "poisoned_values_valid": set(df["poisoned"].unique()) == {0, 1},
        "poisoned_definition_valid": (
            df["poisoned"]
            == (df["attack_type"] != "clean").astype(int)
        ).all()
    })

validation_report = pd.DataFrame(validation_checks)

display(validation_report)

,dataset,attack_type_count,attack_types_correct,missing_attack_type,poisoned_values_valid,poisoned_definition_valid
0,CIC-MalMem-2022,8,True,0,True,True
1,Android,8,True,0,True,True
